#  Detecting and Quantifying Gendered Language in data analyst Job Descriptions

dataset collected from https://www.kaggle.com/datasets/joykimaiyo18/linkedin-data-jobs-dataset

## preparing datasets

goals: 
    - discard missing valued entries


In [1]:
import pandas as pd

# Load your dataset
df = pd.read_csv('linkedin_jobs.csv')


In [2]:
df.head()

,id,title,company,location,link,source,date_posted,work_type,employment_type,description
0,1,Data Analyst,Meta,"New York, NY",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...
1,2,Data Analyst,Meta,"San Francisco, CA",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...
2,3,Data Analyst,Meta,"Los Angeles, CA",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...
3,4,Data Analyst,Meta,"Washington, DC",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-14,NaN,NaN,The Social Measurement team is a growing team ...
4,5,Data Analyst II,Pinterest,"Chicago, IL",https://www.linkedin.com/jobs/view/data-analys...,LinkedIn,2025-04-16,NaN,NaN,About Pinterest\n\nMillions of people around t...


In [3]:
df = df.drop(['location','link','date_posted','work_type','source','employment_type'], axis=1)

In [4]:
df.head()

,id,title,company,description
0,1,Data Analyst,Meta,The Social Measurement team is a growing team ...
1,2,Data Analyst,Meta,The Social Measurement team is a growing team ...
2,3,Data Analyst,Meta,The Social Measurement team is a growing team ...
3,4,Data Analyst,Meta,The Social Measurement team is a growing team ...
4,5,Data Analyst II,Pinterest,About Pinterest\n\nMillions of people around t...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048 entries, 0 to 1047
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           1048 non-null   int64 
 1   title        1048 non-null   object
 2   company      1046 non-null   object
 3   description  1044 non-null   object
dtypes: int64(1), object(3)
memory usage: 32.9+ KB


In [6]:
df.describe(include="O")

,title,company,description
count,1048,1046,1044
unique,523,632,933
top,Data Analyst,Meta,"As a Data Engineer at Meta, you will shape the..."
freq,152,70,11


In [7]:
df = df.dropna(subset=['description']).reset_index(drop=True)

df.info()
df.head(3)

### Gendered lexicon (from Gaucher et al.)

from Appendix A of the paper, we get the list of masculine or feminine words. here is the link : https://www.scribd.com/document/786608715/Gaucher-Friesen-Kay-2011 

In [8]:
masculine_words = [
    'active','adventurous','aggressive','analytical','assertive','athlet','autonom','ambitious',
    'boast','challenging','competitive','confident','couragios','decisive','decision','determined',
    'dominant', 'driven','force','greedy','headstrong','hierarchial','hostil','implusive','independent',
    'individual','intellect','lead','logic','leader','masculine','objective','opinion','outspoken','principle',
    'persist', 'reckless','stubborn','superior', 'self-reliant', 'self-confident','self-sufficient'
]

feminine_words = [
    'affectionate','cheerful','committed','communal','compassionate','connected','considerate','cooperate',
    'dependable','emotional','empathetic','feminine','flatterable',
    'gentle','honest','helpful','interpersonal','interdependent','kind','kinship',
    'loyal','modesty','nurturing','pleasant','polite','quiet','responsive',
    'sensitive','submissive','supportive','sympathetic','tender',
    'together','trust','understanding','warm','whin','yield'
]

len(masculine_words), len(feminine_words)

(42, 38)

In [9]:
import re
import numpy as np

# Precompile patterns
masc_set = set(masculine_words)
fem_set = set(feminine_words)

def label_description(text, delta=0.0005):
    # basic normalization
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    n_tokens = len(tokens) if len(tokens) > 0 else 1

    masc_count = sum(1 for t in tokens if t in masc_set)
    fem_count = sum(1 for t in tokens if t in fem_set)

    masc_norm = masc_count / n_tokens
    fem_norm = fem_count / n_tokens

    diff = masc_norm - fem_norm

    if diff >= delta:
        return 'masculine'
    elif diff <= -delta:
        return 'feminine'
    else:
        return 'neutral'

In [10]:
df['label_auto'] = df['description'].apply(label_description)

df['label_auto'].value_counts()

label_auto
masculine    625
feminine     211
neutral      208
Name: count, dtype: int64

In [11]:
# Sample some descriptions per class for manual inspection
samples_per_class = 10
manual_samples = (
    df.groupby('label_auto', group_keys=False)
      .apply(lambda x: x.sample(min(samples_per_class, len(x)), random_state=42))
)

manual_samples[['title', 'company', 'label_auto', 'description']].head(20)

C:\Users\tunzi\AppData\Local\Temp\ipykernel_17348\2815330638.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(samples_per_class, len(x)), random_state=42))


,title,company,label_auto,description
185,Machine Learning Engineer (Junior),LogicMatrix,feminine,Machine Learning Engineer (Junior)\n\n\n\nA Bi...
858,Cleaner,A & J Fencing,feminine,**Overview** \nWe are seeking a dedicated and...
761,Data Visualization Analyst,Weekday (YC W21),feminine,This role is for one of our clients\n\nIndustr...
347,TikTok Shop - Data Analyst,TikTok,feminine,Responsibilities\n\nThe commerce industry has ...
305,"Data Engineer I, SCOT - AIM",Amazon,feminine,Description\n\nSCOT's Automated Inventory Mana...
1030,Software Engineer - Android,Blinq,feminine,"**What is Blinq?** \n\n\n\n\n\n At Blinq, we b..."
270,"Sr. Data Engineer, Analytics",Quizlet,feminine,About Quizlet\n\nInspired by our belief that a...
913,Senior AI Agent Engineer,Workday,feminine,"Your work days are brighter here.\nAt Workday,..."
54,Data Analyst Contractor,Instacart,feminine,We're transforming the grocery industry\n\nAt ...
546,Software Engineer (L4) - Cloud Network Enginee...,Netflix,feminine,Netflix is one of the world's leading entertai...


### Train/validation/test split

In [12]:
from sklearn.model_selection import train_test_split

X = df['description'].values
y = df['label_auto'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

len(X_train), len(X_val), len(X_test)

(730, 157, 157)

### Baseline model: Logistic regression

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

baseline_clf = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3
    )),
    ('logreg', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        multi_class='ovr'
    ))
])

baseline_clf.fit(X_train, y_train)

y_pred_val = baseline_clf.predict(X_val)
print(classification_report(y_val, y_pred_val))

C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

    feminine       0.51      0.65      0.57        31
   masculine       0.80      0.74      0.77        94
     neutral       0.50      0.47      0.48        32

    accuracy                           0.67       157
   macro avg       0.60      0.62      0.61       157
weighted avg       0.68      0.67      0.67       157



In [14]:
from sklearn.metrics import confusion_matrix

y_pred_test = baseline_clf.predict(X_test)
print(classification_report(y_test, y_pred_test))

cm = confusion_matrix(y_test, y_pred_test, labels=['masculine', 'feminine', 'neutral'])
cm

              precision    recall  f1-score   support

    feminine       0.54      0.59      0.57        32
   masculine       0.81      0.73      0.77        94
     neutral       0.43      0.52      0.47        31

    accuracy                           0.66       157
   macro avg       0.60      0.61      0.60       157
weighted avg       0.68      0.66      0.67       157



array([[69, 11, 14],
       [ 6, 19,  7],
       [10,  5, 16]])

In [15]:
df['label_auto'].value_counts(normalize=False)
df['label_auto'].value_counts(normalize=True)

label_auto
masculine    0.598659
feminine     0.202107
neutral      0.199234
Name: proportion, dtype: float64

In [16]:
from sklearn.metrics import classification_report
print(classification_report(y_val, y_pred_val))

              precision    recall  f1-score   support

    feminine       0.51      0.65      0.57        31
   masculine       0.80      0.74      0.77        94
     neutral       0.50      0.47      0.48        32

    accuracy                           0.67       157
   macro avg       0.60      0.62      0.61       157
weighted avg       0.68      0.67      0.67       157



### Advanced model: RoBERTa

In [17]:
!pip install transformers torch --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import torch
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

label2id = {'masculine': 0, 'feminine': 1, 'neutral': 2}
id2label = {v: k for k, v in label2id.items()}

y_train_ids = [label2id[label] for label in y_train]
y_val_ids = [label2id[label] for label in y_val]
y_test_ids = [label2id[label] for label in y_test]

tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')

In [19]:
from torch.utils.data import Dataset

class JobAdsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item


In [20]:
train_dataset = JobAdsDataset(X_train, y_train_ids, tokenizer)
val_dataset   = JobAdsDataset(X_val,   y_val_ids,   tokenizer)
test_dataset  = JobAdsDataset(X_test,  y_test_ids,  tokenizer)

len(train_dataset), len(val_dataset), len(test_dataset)


(730, 157, 157)

In [21]:
from torch.utils.data import DataLoader

batch_size = 8  # increase if your machine can handle it

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

In [22]:
batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, v.shape)

input_ids torch.Size([8, 256])
attention_mask torch.Size([8, 256])
labels torch.Size([8])


In [23]:
from transformers import RobertaForSequenceClassification

num_labels = 3

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

device


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


device(type='cpu')

In [24]:
from transformers import get_linear_schedule_with_warmup
import torch.nn as nn

epochs = 3
learning_rate = 2e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

loss_fn = nn.CrossEntropyLoss()


In [25]:
def train_one_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        ) 
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


def eval_one_epoch(model, data_loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(data_loader)
    return avg_loss, all_labels, all_preds


In [26]:
from sklearn.metrics import classification_report

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, y_val_true_ids, y_val_pred_ids = eval_one_epoch(model, val_loader, device)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Train loss: {train_loss:.4f}")
    print(f"  Val   loss: {val_loss:.4f}")

    print(classification_report(
        y_val_true_ids,
        y_val_pred_ids,
        target_names=['masculine', 'feminine', 'neutral']
    ))


Epoch 1/3
  Train loss: 0.9550
  Val   loss: 0.9428
              precision    recall  f1-score   support

   masculine       0.60      1.00      0.75        94
    feminine       0.00      0.00      0.00        31
     neutral       0.00      0.00      0.00        32

    accuracy                           0.60       157
   macro avg       0.20      0.33      0.25       157
weighted avg       0.36      0.60      0.45       157



C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Epoch 2/3
  Train loss: 0.8980
  Val   loss: 0.8992
              precision    recall  f1-score   support

   masculine       0.60      0.98      0.74        94
    feminine       0.00      0.00      0.00        31
     neutral       0.50      0.06      0.11        32

    accuracy                           0.60       157
   macro avg       0.37      0.35      0.29       157
weighted avg       0.46      0.60      0.47       157



C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Epoch 3/3
  Train loss: 0.8412
  Val   loss: 0.8571
              precision    recall  f1-score   support

   masculine       0.61      0.94      0.74        94
    feminine       0.00      0.00      0.00        31
     neutral       0.50      0.19      0.27        32

    accuracy                           0.60       157
   macro avg       0.37      0.37      0.34       157
weighted avg       0.47      0.60      0.50       157



C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [27]:
test_loss, y_test_true_ids, y_test_pred_ids = eval_one_epoch(model, test_loader, device)
print(f"Test loss: {test_loss:.4f}")

print(classification_report(
    y_test_true_ids,
    y_test_pred_ids,
    target_names=['masculine', 'feminine', 'neutral']
))

Test loss: 0.8859
              precision    recall  f1-score   support

   masculine       0.61      0.94      0.74        94
    feminine       0.00      0.00      0.00        32
     neutral       0.50      0.19      0.28        31

    accuracy                           0.60       157
   macro avg       0.37      0.38      0.34       157
weighted avg       0.46      0.60      0.50       157



C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tunzi\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo